In [1]:
import os
import networkx as nx
import numpy as np
import pandas as pd
import random
from flow_utils import *

from fugu.scaffold.scaffold import Scaffold
from fugu.bricks.bricks import Brick
from fugu.backends.snn_backend import snn_Backend

# Get the absolute path of the current script
os.getcwd()

'/Users/kamerongano/Documents/GitHub/Fugu_dev/fugu/nscale_flow'

In [2]:
# network settings
# 200 neurons, 20% connectivity, 1 weight, bias of 1, threshold of 100, run for 500 time steps
PATH_TO_NSCALE = "~/Documents/Github/nscale/configs/distinct_demo/"
SANA_FE = True
num_neurons = 200
bias = 1
th = 100
prob_w = 0.2
w = 1
time = 500

In [3]:
# connectivity matrix
seed = 42
random.seed(seed)
np.random.seed(seed)

connec_matrix = []
for i in range(num_neurons):
    connec_vector = []
    for j in range(num_neurons):
        if random.random() < prob_w:
            connec_vector.append(1)
        else:
            connec_vector.append(0)
    connec_matrix.append(connec_vector)
#connec_matrix

In [4]:
# making a brick based off of the above settings
class recurrent(Brick):
    def __init__(self, name=None):
        super().__init__()
        self.is_built = False
        self.metadata = {'D':1}  
        self.name = name
    def build(self,
             graph,
             metadata,
             control_nodes,
             input_lists,
             input_codings):

        output_codings = []

        #We also, obviously, need to build the computational portion of our graph
        for i in range(num_neurons):
            graph.add_node(self.name + '_' + str(i),
                           index=i,
                           threshold=th,
                           decay=0.0,
                           bias=bias,
                           p=1.0,
                           potential=0.0
                          )
        for i in range(num_neurons):
            for j in range(num_neurons):
                if connec_matrix[i][j] == 1:
                    graph.add_edge(self.name + '_' + str(i),
                                  self.name + '_' + str(j),
                                  weight=w,
                                  delay=1.0)
        self.is_built=True
        
        # bricks can have more than one output
        output_lists = []
        
        return (graph,
               self.metadata,
                [],
                output_lists,
                output_codings
               )

In [5]:
scaffold = Scaffold()
scaffold.add_brick(recurrent(name='recurrent'), [], output=True)
scaffold.lay_bricks()
#scaffold.summary(verbose=2)

In [6]:
if SANA_FE:
    # get sanafe backend and compile the scaffold
    # TODO: move arch YAML file to fugu
    from sanafe import fugu as sf
    ARCH_FILENAME = "/Users/kamerongano/Documents/Github/SANA-FE/arch/neuroscale.yaml"
    MAPPINGS = "core_mappings.json"
    #backend = snn_Backend()
    backend = sf.sanafe_Backend()
    backend_args = {}
    backend_args['record'] = 'all'
    backend_args['arch'] = ARCH_FILENAME
    backend_args['mappings'] = MAPPINGS
    #backend_args['debug_mode'] = True # show the optential at each time
    backend.compile(scaffold, backend_args)
    
else:
    # use regular snn backend
    backend = snn_Backend()
    backend_args = {}
    backend_args['record'] = 'all'
    #backend_args['debug_mode'] = True # show the optential at each time
    backend.compile(scaffold, backend_args)

fugu:{'index': 0, 'threshold': 100, 'decay': 0.0, 'bias': 1, 'p': 1.0, 'potential': 0.0, 'brick': 'Brick-0', 'neuron_number': 0}
sanafe_props:{'threshold': 100, 'leak_decay': 1.0, 'bias': 1, 'potential': 0.0}
Enabling logging for neuron recurrent_0
fugu:{'index': 1, 'threshold': 100, 'decay': 0.0, 'bias': 1, 'p': 1.0, 'potential': 0.0, 'brick': 'Brick-0', 'neuron_number': 1}
sanafe_props:{'threshold': 100, 'leak_decay': 1.0, 'bias': 1, 'potential': 0.0}
Enabling logging for neuron recurrent_1
fugu:{'index': 2, 'threshold': 100, 'decay': 0.0, 'bias': 1, 'p': 1.0, 'potential': 0.0, 'brick': 'Brick-0', 'neuron_number': 2}
sanafe_props:{'threshold': 100, 'leak_decay': 1.0, 'bias': 1, 'potential': 0.0}
Enabling logging for neuron recurrent_2
fugu:{'index': 3, 'threshold': 100, 'decay': 0.0, 'bias': 1, 'p': 1.0, 'potential': 0.0, 'brick': 'Brick-0', 'neuron_number': 3}
sanafe_props:{'threshold': 100, 'leak_decay': 1.0, 'bias': 1, 'potential': 0.0}
Enabling logging for neuron recurrent_3
fugu

In [7]:
result = backend.run(time)
# type(result) # pandas.core.frame.DataFrame

/Users/kamerongano/Documents/Github/SANA-FE/arch/neuroscale.yaml
Executed steps: [500/500]


In [8]:
matrix = result.to_numpy()
spike_times = {}
for i in range(num_neurons):
    spike_times[i] = []
for i in matrix:
    spike_times[i[1]].append(int(i[0]))

In [9]:
# record the spikes into spikes_fugu.json file
import json
with open('spikes_fugu.json', 'w') as file:
    json.dump(spike_times, file)

In [10]:
generate_act_network_assets()

successfully generate act_json/act_network.json


{'network': {'ticks': 500,
  'neuron_dict': {'0': {'fan-in': [3,
     5,
     10,
     19,
     20,
     22,
     25,
     28,
     38,
     42,
     43,
     47,
     53,
     56,
     61,
     71,
     77,
     79,
     83,
     84,
     94,
     106,
     111,
     114,
     115,
     119,
     120,
     121,
     124,
     125,
     137,
     139,
     141,
     144,
     155,
     160,
     161,
     165,
     167,
     178],
    'fan-out': [1,
     7,
     9,
     12,
     13,
     19,
     23,
     26,
     27,
     41,
     44,
     46,
     56,
     58,
     67,
     88,
     89,
     90,
     94,
     100,
     106,
     113,
     119,
     124,
     126,
     131,
     134,
     136,
     139,
     154,
     160,
     165,
     169,
     175,
     176,
     183,
     188,
     197],
    'th': 100,
    'leak': 0.0,
    'bias': 1,
    'refractory period': 0,
    'prob': 1.0,
    'neuron_id': 0},
   '1': {'fan-in': [0,
     7,
     8,
     11,
     17,
     19,
     23,
     27